# NOVA RETAIL — Despacho vs Recepción

## Análisis Forense de Variación en la Cadena de Suministro
**Proyecto:** Diagnóstico de Prevención de Pérdidas  
**Autor:** Denz One  
**Fase:** Reconciliación operativa entre CEDIS y tiendas  
**Objetivo:** Comparar los registros de despacho del CEDIS contra las recepciones reportadas por tienda para identificar discrepancias sistemáticas incompatibles con error operativo aleatorio.

---

### Propósito de este notebook
Este notebook evalúa la integridad del flujo de mercancía entre el centro de distribución y las tiendas. Su función es detectar:

- discrepancias entre lo despachado y lo recibido
- patrones concentrados por tienda, categoría o valor
- anomalías horarias en la recepción de mercancía
- señales de pérdida selectiva en productos de alto valor

### Datasets involucrados
- `dispatches.csv`
- `receptions.csv`
- `products_sap.csv`
- `stores.csv`

### Nota del analista
La diferencia entre despacho y recepción no prueba fraude por sí sola.  
Pero cuando la variación se concentra en ciertas tiendas, productos y horarios, deja de parecer ruido operativo y se convierte en una prioridad investigativa.

In [101]:
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

DATA_PATH = Path("../data/raw")

dispatches = pd.read_csv(DATA_PATH / "dispatches.csv")
receptions = pd.read_csv(DATA_PATH / "receptions.csv")
products_sap = pd.read_csv(DATA_PATH / "products_sap.csv")
stores = pd.read_csv(DATA_PATH / "stores.csv")

print("Datasets cargados correctamente.")
print("dispatches:", dispatches.shape)
print("receptions:", receptions.shape)
print("products_sap:", products_sap.shape)
print("stores:", stores.shape)

Datasets cargados correctamente.
dispatches: (265964, 11)
receptions: (265964, 11)
products_sap: (14000, 6)
stores: (187, 15)


In [102]:
dispatches.head()

,dispatch_id,store_id,sku,quantity_dispatched,dispatch_date,dispatch_time,route,truck_id,driver_id,authorized_by,cedis_system_record
0,DSP-00000001,1,SAP-0002901,2,2024-01-02,03:47,SUR,TRUCK-009,DRV-016,RCISNE04,SAP
1,DSP-00000002,1,SAP-0003144,3,2024-01-02,05:06,SUR,TRUCK-022,DRV-035,RCISNE04,SAP
2,DSP-00000003,1,SAP-0008232,42,2024-01-02,04:02,SUR,TRUCK-001,DRV-006,RCISNE04,SAP
3,DSP-00000004,1,SAP-0003856,4,2024-01-02,05:38,SUR,TRUCK-001,DRV-036,RCISNE04,SAP
4,DSP-00000005,1,SAP-0008046,50,2024-01-02,05:44,SUR,TRUCK-018,DRV-027,RCISNE04,SAP


In [103]:
receptions.head()

,reception_id,store_id,sku,quantity_received,unit_type,reception_date_str,reception_time,received_by,dispatch_reference,system_source,notes
0,REC-00000001,1,SAP-0002901,1,CAJA,03/01/2024,07:47,NaN,DSP-00000001,SAP,OK
1,REC-00000002,1,SAP-0003144,1,CAJA,02/01/2024,08:02,FLÓRE82,DSP-00000002,SAP,OK
2,REC-00000003,1,SAP-0008232,42,UNIDAD,03/01/2024,09:44,NaN,DSP-00000003,SAP,NaN
3,REC-00000004,1,SAP-0003856,4,UNIDAD,01/03/2024,07:44,NaN,DSP-00000004,SAP,caja dañada
4,REC-00000005,1,SAP-0008046,50,UNIDAD,01/03/2024,08:06,CARRI84,DSP-00000005,SAP,revisar con proveedor


In [104]:
summary = pd.DataFrame({
    "dataset": ["dispatches", "receptions", "products_sap", "stores"],
    "filas": [
        dispatches.shape[0],
        receptions.shape[0],
        products_sap.shape[0],
        stores.shape[0]
    ],
    "columnas": [
        dispatches.shape[1],
        receptions.shape[1],
        products_sap.shape[1],
        stores.shape[1]
    ]
})

summary

,dataset,filas,columnas
0,dispatches,265964,11
1,receptions,265964,11
2,products_sap,14000,6
3,stores,187,15


In [105]:
high_value_categories = ["ELECTRONICA", "TELEFONIA", "COMPUTO", "ELECTRODOMESTICOS"]

products_priority = products_sap[
    products_sap["category_sap"].isin(high_value_categories)
].copy()

products_priority = products_priority[
    products_priority["price_sap"] >= products_priority["price_sap"].quantile(0.95)
].copy()

print("SKUs prioritarios:", len(products_priority))
products_priority[["sku_sap", "description_sap", "category_sap", "price_sap"]].head(10)

SKUs prioritarios: 315


,sku_sap,description_sap,category_sap,price_sap
10,SAP-0000011,"TABLET TCL 75""",ELECTRONICA,34045.59
11,SAP-0000012,"MONITOR HISENSE 32""",ELECTRONICA,33475.93
14,SAP-0000015,MONITOR LG 512GB,ELECTRONICA,33246.01
25,SAP-0000026,CONSOLA JBL 128GB,ELECTRONICA,32845.31
28,SAP-0000029,"TELEVISOR JBL 75""",ELECTRONICA,32747.95
44,SAP-0000045,"TABLET SAMSUNG 65""",ELECTRONICA,32276.51
51,SAP-0000052,"TABLET LG 32""",ELECTRONICA,31928.65
53,SAP-0000054,"TELEVISOR JBL 32""",ELECTRONICA,34996.96
70,SAP-0000071,"TELEVISOR TCL 43""",ELECTRONICA,33270.48
76,SAP-0000077,CONSOLA JBL 512GB,ELECTRONICA,33978.82


In [106]:
dispatches_priority = dispatches.merge(
    products_priority[["sku_sap", "category_sap", "price_sap"]],
    left_on="sku",
    right_on="sku_sap",
    how="inner"
)

receptions_priority = receptions.merge(
    products_priority[["sku_sap", "category_sap", "price_sap"]],
    left_on="sku",
    right_on="sku_sap",
    how="inner"
)

print("Despachos prioritarios:", dispatches_priority.shape)
print("Recepciones prioritarias:", receptions_priority.shape)

Despachos prioritarios: (5989, 14)
Recepciones prioritarias: (5989, 14)


In [107]:
receptions_priority["unit_type"].value_counts()

unit_type
UNIDAD    4505
CAJA      1484
Name: count, dtype: int64

In [108]:
receptions_priority_box_named = receptions_priority_box.merge(
    products_sap[["sku_sap", "description_sap"]],
    left_on="sku",
    right_on="sku_sap",
    how="left"
)

receptions_priority_box_named[[
    "sku",
    "description_sap",
    "category_sap",
    "price_sap",
    "quantity_received",
    "unit_type",
    "notes"
]].head(40)

,sku,description_sap,category_sap,price_sap,quantity_received,unit_type,notes
0,SAP-0001967,MONITOR LG 256GB,ELECTRONICA,33812.29,1,CAJA,NaN
1,SAP-0001161,TELEVISOR TCL 256GB,ELECTRONICA,34856.60,1,CAJA,NaN
2,SAP-0005322,"USB ACER 43""",COMPUTO,33558.35,1,CAJA,NaN
3,SAP-0002060,"BOCINA SAMSUNG 65""",ELECTRONICA,33602.16,1,CAJA,NaN
4,SAP-0005226,"LAPTOP DELL 75""",COMPUTO,32925.41,1,CAJA,NaN
...,...,...,...,...,...,...,...
35,SAP-0001746,CAMARA LG 512GB,ELECTRONICA,32770.16,1,CAJA,faltante parcial
36,SAP-0001350,AUDIFONOS HISENSE 512GB,ELECTRONICA,32846.78,1,CAJA,NaN
37,SAP-0001014,TELEVISOR SAMSUNG 256GB,ELECTRONICA,33431.63,1,CAJA,NaN
38,SAP-0005486,LAPTOP LENOVO 256GB,COMPUTO,32964.52,1,CAJA,producto sin codigo en sistema


In [109]:
receptions_priority_box_named[[
    "description_sap",
    "category_sap",
    "price_sap",
    "quantity_received",
    "unit_type",
    "notes"
]].sort_values("price_sap", ascending=False).head(40)

,description_sap,category_sap,price_sap,quantity_received,unit_type,notes
680,MONITOR HISENSE 512GB,ELECTRONICA,34998.49,1,CAJA,caja dañada
874,MONITOR HISENSE 512GB,ELECTRONICA,34998.49,1,CAJA,caja dañada
237,MONITOR HISENSE 512GB,ELECTRONICA,34998.49,1,CAJA,NaN
755,MONITOR HISENSE 512GB,ELECTRONICA,34998.49,1,CAJA,NaN
1088,LAPTOP LOGITECH 256GB,COMPUTO,34987.12,1,CAJA,faltante parcial
...,...,...,...,...,...,...
258,"WEBCAM HP 65""",COMPUTO,34968.49,1,CAJA,faltante parcial
169,"WEBCAM HP 65""",COMPUTO,34968.49,1,CAJA,caja dañada
1328,WEBCAM ACER 128GB,COMPUTO,34956.41,1,CAJA,faltante parcial
957,WEBCAM ACER 128GB,COMPUTO,34956.41,1,CAJA,NaN


In [110]:
merged_priority = dispatches_priority.merge(
    receptions_priority,
    left_on="dispatch_id",
    right_on="dispatch_reference",
    how="inner",
    suffixes=("_dispatch", "_reception")
)

# Regla operativa:
# En el universo prioritario, CAJA se trata como unidad individual
merged_priority["quantity_received_normalized"] = merged_priority["quantity_received"]

merged_priority[[
    "dispatch_id",
    "sku_dispatch",
    "category_sap_dispatch",
    "price_sap_dispatch",
    "quantity_dispatched",
    "quantity_received",
    "unit_type",
    "quantity_received_normalized",
    "notes"
]].head(20)

,dispatch_id,sku_dispatch,category_sap_dispatch,price_sap_dispatch,quantity_dispatched,quantity_received,unit_type,quantity_received_normalized,notes
0,DSP-00000158,SAP-0000658,ELECTRONICA,31792.16,8,8,UNIDAD,8,caja dañada
1,DSP-00000178,SAP-0000848,ELECTRONICA,34001.04,3,3,UNIDAD,3,OK
2,DSP-00000180,SAP-0005199,COMPUTO,34574.53,3,3,UNIDAD,3,producto sin codigo en sistema
3,DSP-00000242,SAP-0000795,ELECTRONICA,33584.20,1,1,UNIDAD,1,revisar con proveedor
4,DSP-00000248,SAP-0005513,COMPUTO,34557.23,3,3,UNIDAD,3,OK
5,DSP-00000293,SAP-0001409,ELECTRONICA,32725.71,9,9,UNIDAD,9,caja dañada
6,DSP-00000326,SAP-0001967,ELECTRONICA,33812.29,5,1,CAJA,1,NaN
7,DSP-00000327,SAP-0001621,ELECTRONICA,32330.85,4,4,UNIDAD,4,NaN
8,DSP-00000352,SAP-0001033,ELECTRONICA,33347.33,5,5,UNIDAD,5,caja dañada
9,DSP-00000377,SAP-0001661,ELECTRONICA,34563.79,6,6,UNIDAD,6,caja dañada


In [111]:
merged_priority["quantity_diff"] = (
    merged_priority["quantity_dispatched"] - merged_priority["quantity_received_normalized"]
)

merged_priority["pct_diff"] = np.where(
    merged_priority["quantity_dispatched"] > 0,
    (merged_priority["quantity_diff"] / merged_priority["quantity_dispatched"]) * 100,
    np.nan
)

merged_priority[[
    "dispatch_id",
    "sku_dispatch",
    "quantity_dispatched",
    "quantity_received_normalized",
    "quantity_diff",
    "pct_diff",
    "notes"
]].sort_values("pct_diff", ascending=False).head(30)

,dispatch_id,sku_dispatch,quantity_dispatched,quantity_received_normalized,quantity_diff,pct_diff,notes
380,DSP-00016792,SAP-0000299,12,1,11,91.666667,producto sin codigo en sistema
3497,DSP-00154234,SAP-0000232,12,1,11,91.666667,NaN
2194,DSP-00097141,SAP-0005981,12,1,11,91.666667,NaN
1590,DSP-00070189,SAP-0000971,12,1,11,91.666667,NaN
259,DSP-00011966,SAP-0006035,12,1,11,91.666667,producto sin codigo en sistema
...,...,...,...,...,...,...,...
5769,DSP-00256468,SAP-0005604,12,1,11,91.666667,OK
3710,DSP-00164699,SAP-0006182,12,1,11,91.666667,producto sin codigo en sistema
2173,DSP-00096142,SAP-0000026,12,1,11,91.666667,OK
465,DSP-00020200,SAP-0005633,12,1,11,91.666667,NaN


In [112]:
def normalize_priority_quantity(row):
    if row["unit_type"] == "CAJA":
        if row["quantity_received"] == 1 and row["quantity_dispatched"] in [6, 12, 24]:
            return row["quantity_dispatched"]
        else:
            return row["quantity_received"]
    return row["quantity_received"]

merged_priority["quantity_received_normalized_v2"] = merged_priority.apply(
    normalize_priority_quantity,
    axis=1
)

merged_priority["quantity_diff_v2"] = (
    merged_priority["quantity_dispatched"] - merged_priority["quantity_received_normalized_v2"]
)

merged_priority["pct_diff_v2"] = np.where(
    merged_priority["quantity_dispatched"] > 0,
    (merged_priority["quantity_diff_v2"] / merged_priority["quantity_dispatched"]) * 100,
    np.nan
)

merged_priority[[
    "dispatch_id",
    "sku_dispatch",
    "quantity_dispatched",
    "quantity_received",
    "unit_type",
    "quantity_received_normalized_v2",
    "quantity_diff_v2",
    "pct_diff_v2",
    "notes"
]].sort_values("pct_diff_v2", ascending=False).head(30)

,dispatch_id,sku_dispatch,quantity_dispatched,quantity_received,unit_type,quantity_received_normalized_v2,quantity_diff_v2,pct_diff_v2,notes
2978,DSP-00132219,SAP-0001717,11,1,CAJA,1,10,90.909091,NaN
1915,DSP-00085032,SAP-0000544,11,1,CAJA,1,10,90.909091,NaN
516,DSP-00022066,SAP-0001556,11,1,CAJA,1,10,90.909091,OK
1845,DSP-00081379,SAP-0001656,11,1,CAJA,1,10,90.909091,NaN
368,DSP-00016338,SAP-0000756,11,1,CAJA,1,10,90.909091,NaN
...,...,...,...,...,...,...,...,...,...
4742,DSP-00211119,SAP-0000745,11,1,CAJA,1,10,90.909091,caja dañada
1693,DSP-00074871,SAP-0002023,11,1,CAJA,1,10,90.909091,NaN
5517,DSP-00244541,SAP-0001120,11,1,CAJA,1,10,90.909091,faltante parcial
1696,DSP-00075434,SAP-0000725,11,1,CAJA,1,10,90.909091,NaN


In [113]:
def normalize_priority_quantity_v3(row):
    if row["unit_type"] == "CAJA":
        if row["quantity_received"] == 1 and row["quantity_dispatched"] > 1:
            return row["quantity_dispatched"]
        else:
            return row["quantity_received"]
    return row["quantity_received"]

merged_priority["quantity_received_normalized_v3"] = merged_priority.apply(
    normalize_priority_quantity_v3,
    axis=1
)

merged_priority["quantity_diff_v3"] = (
    merged_priority["quantity_dispatched"] - merged_priority["quantity_received_normalized_v3"]
)

merged_priority["pct_diff_v3"] = np.where(
    merged_priority["quantity_dispatched"] > 0,
    (merged_priority["quantity_diff_v3"] / merged_priority["quantity_dispatched"]) * 100,
    np.nan
)

merged_priority[[
    "dispatch_id",
    "sku_dispatch",
    "quantity_dispatched",
    "quantity_received",
    "unit_type",
    "quantity_received_normalized_v3",
    "quantity_diff_v3",
    "pct_diff_v3",
    "notes"
]].sort_values("pct_diff_v3", ascending=False).head(30)

,dispatch_id,sku_dispatch,quantity_dispatched,quantity_received,unit_type,quantity_received_normalized_v3,quantity_diff_v3,pct_diff_v3,notes
3837,DSP-00170355,SAP-0000950,12,2,CAJA,2,10,83.333333,revisar con proveedor
400,DSP-00017591,SAP-0000106,12,2,CAJA,2,10,83.333333,OK
4783,DSP-00213093,SAP-0005730,12,2,CAJA,2,10,83.333333,revisar con proveedor
2869,DSP-00127669,SAP-0000993,12,2,CAJA,2,10,83.333333,NaN
1982,DSP-00088014,SAP-0000649,12,2,CAJA,2,10,83.333333,revisar con proveedor
...,...,...,...,...,...,...,...,...,...
3100,DSP-00137288,SAP-0001033,12,2,CAJA,2,10,83.333333,NaN
686,DSP-00030885,SAP-0001691,12,2,CAJA,2,10,83.333333,NaN
3275,DSP-00144165,SAP-0000071,12,2,CAJA,2,10,83.333333,NaN
3986,DSP-00178022,SAP-0000015,12,2,CAJA,2,10,83.333333,producto sin codigo en sistema


In [114]:
merged_priority_named = merged_priority.merge(
    products_sap[["sku_sap", "description_sap"]],
    left_on="sku_dispatch",
    right_on="sku_sap",
    how="left"
)

merged_priority_named[[
    "dispatch_id",
    "description_sap",
    "category_sap_dispatch",
    "price_sap_dispatch",
    "quantity_dispatched",
    "quantity_received",
    "unit_type",
    "notes"
]].head(50)

,dispatch_id,description_sap,category_sap_dispatch,price_sap_dispatch,quantity_dispatched,quantity_received,unit_type,notes
0,DSP-00000158,"CAMARA HISENSE 75""",ELECTRONICA,31792.16,8,8,UNIDAD,caja dañada
1,DSP-00000178,CONSOLA LG 128GB,ELECTRONICA,34001.04,3,3,UNIDAD,OK
2,DSP-00000180,"MOUSE LENOVO 55""",COMPUTO,34574.53,3,3,UNIDAD,producto sin codigo en sistema
3,DSP-00000242,MONITOR LG 256GB,ELECTRONICA,33584.20,1,1,UNIDAD,revisar con proveedor
4,DSP-00000248,"WEBCAM LENOVO 32""",COMPUTO,34557.23,3,3,UNIDAD,OK
...,...,...,...,...,...,...,...,...
45,DSP-00001842,AUDIFONOS HISENSE 512GB,ELECTRONICA,32846.78,4,4,UNIDAD,OK
46,DSP-00001847,"CAMARA SONY 43""",ELECTRONICA,33112.49,7,7,UNIDAD,revisar con proveedor
47,DSP-00001851,"DESKTOP HP 43""",COMPUTO,33177.93,9,9,UNIDAD,revisar con proveedor
48,DSP-00001854,"USB LENOVO 65""",COMPUTO,33355.28,8,8,UNIDAD,NaN


In [115]:
merged_priority_named = merged_priority.merge(
    products_sap[["sku_sap", "description_sap"]],
    left_on="sku_dispatch",
    right_on="sku_sap",
    how="left"
)

In [116]:
high_diff_raw = merged_priority_named[
    (merged_priority_named["unit_type"] == "CAJA") &
    (merged_priority_named["quantity_received"] == 1) &
    (merged_priority_named["quantity_dispatched"] > 1)
].copy()

In [117]:
high_diff_raw[[
    "dispatch_id",
    "description_sap",
    "category_sap_dispatch",
    "price_sap_dispatch",
    "quantity_dispatched",
    "quantity_received",
    "unit_type",
    "notes"
]].head(100)

,dispatch_id,description_sap,category_sap_dispatch,price_sap_dispatch,quantity_dispatched,quantity_received,unit_type,notes
6,DSP-00000326,MONITOR LG 256GB,ELECTRONICA,33812.29,5,1,CAJA,NaN
11,DSP-00000465,TELEVISOR TCL 256GB,ELECTRONICA,34856.60,2,1,CAJA,NaN
12,DSP-00000503,"USB ACER 43""",COMPUTO,33558.35,4,1,CAJA,NaN
18,DSP-00000726,"TECLADO ASUS 43""",COMPUTO,33751.29,6,1,CAJA,faltante parcial
25,DSP-00000919,USB DELL 512GB,COMPUTO,32710.04,7,1,CAJA,producto sin codigo en sistema
...,...,...,...,...,...,...,...,...
444,DSP-00019500,PANTALLA TCL 256GB,ELECTRONICA,34271.57,7,1,CAJA,caja dañada
446,DSP-00019539,"DESKTOP HP 65""",COMPUTO,34896.39,2,1,CAJA,NaN
451,DSP-00019701,DISCO DURO DELL 128GB,COMPUTO,34131.49,4,1,CAJA,NaN
460,DSP-00019935,"AUDIFONOS SONY 43""",ELECTRONICA,32064.34,7,1,CAJA,caja dañada


In [118]:
high_diff_raw["notes"].value_counts(dropna=False)

notes
NaN                               673
revisar con proveedor             148
caja dañada                       131
OK                                125
faltante parcial                  122
producto sin codigo en sistema    121
Name: count, dtype: int64

In [119]:
def classify_anomaly_type(row):
    note = str(row["notes"]).strip().lower()

    if "producto sin codigo" in note:
        return "CATALOGO_INVISIBLE"
    elif "faltante parcial" in note:
        return "INCIDENCIA_OPERATIVA_FALTANTE"
    elif "caja dañada" in note:
        return "INCIDENCIA_OPERATIVA_DAÑO"
    elif "revisar con proveedor" in note:
        return "INCIDENCIA_PROVEEDOR"
    elif row["unit_type"] == "CAJA" and row["quantity_received"] == 1 and row["quantity_dispatched"] > 1:
        return "PROBABLE_ERROR_UOM"
    else:
        return "OTRO"

high_diff_raw["anomaly_type"] = high_diff_raw.apply(classify_anomaly_type, axis=1)

high_diff_raw["anomaly_type"].value_counts()

anomaly_type
PROBABLE_ERROR_UOM               798
INCIDENCIA_PROVEEDOR             148
INCIDENCIA_OPERATIVA_DAÑO        131
INCIDENCIA_OPERATIVA_FALTANTE    122
CATALOGO_INVISIBLE               121
Name: count, dtype: int64

In [120]:
high_diff_raw[[
    "dispatch_id",
    "description_sap",
    "quantity_dispatched",
    "quantity_received",
    "unit_type",
    "notes",
    "anomaly_type"
]].head(50)

,dispatch_id,description_sap,quantity_dispatched,quantity_received,unit_type,notes,anomaly_type
6,DSP-00000326,MONITOR LG 256GB,5,1,CAJA,NaN,PROBABLE_ERROR_UOM
11,DSP-00000465,TELEVISOR TCL 256GB,2,1,CAJA,NaN,PROBABLE_ERROR_UOM
12,DSP-00000503,"USB ACER 43""",4,1,CAJA,NaN,PROBABLE_ERROR_UOM
18,DSP-00000726,"TECLADO ASUS 43""",6,1,CAJA,faltante parcial,INCIDENCIA_OPERATIVA_FALTANTE
25,DSP-00000919,USB DELL 512GB,7,1,CAJA,producto sin codigo en sistema,CATALOGO_INVISIBLE
...,...,...,...,...,...,...,...
198,DSP-00009362,CONSOLA JBL 128GB,4,1,CAJA,OK,PROBABLE_ERROR_UOM
206,DSP-00009677,"IMPRESORA HP 65""",5,1,CAJA,producto sin codigo en sistema,CATALOGO_INVISIBLE
207,DSP-00009701,CONSOLA JBL 128GB,10,1,CAJA,NaN,PROBABLE_ERROR_UOM
216,DSP-00010222,"DESKTOP LENOVO 43""",12,1,CAJA,caja dañada,INCIDENCIA_OPERATIVA_DAÑO


In [121]:
risk_by_store = (
    high_diff_raw
    .groupby(["store_id_dispatch", "anomaly_type"])
    .size()
    .reset_index(name="casos")
)

risk_by_store.sort_values(["casos"], ascending=False).head(30)

,store_id_dispatch,anomaly_type,casos
444,147,PROBABLE_ERROR_UOM,11
296,97,PROBABLE_ERROR_UOM,9
172,55,PROBABLE_ERROR_UOM,9
491,162,PROBABLE_ERROR_UOM,9
224,72,PROBABLE_ERROR_UOM,9
...,...,...,...
430,141,PROBABLE_ERROR_UOM,6
393,130,PROBABLE_ERROR_UOM,6
379,125,PROBABLE_ERROR_UOM,6
442,146,PROBABLE_ERROR_UOM,6


In [122]:
risk_pivot = risk_by_store.pivot(
    index="store_id_dispatch",
    columns="anomaly_type",
    values="casos"
).fillna(0)

risk_pivot["total_anomalias"] = risk_pivot.sum(axis=1)
risk_pivot = risk_pivot.sort_values("total_anomalias", ascending=False)

risk_pivot.head(20)

anomaly_type,CATALOGO_INVISIBLE,INCIDENCIA_OPERATIVA_DAÑO,INCIDENCIA_OPERATIVA_FALTANTE,INCIDENCIA_PROVEEDOR,PROBABLE_ERROR_UOM,total_anomalias
store_id_dispatch,,,,,,
138,0.0,1.0,2.0,2.0,9.0,14.0
55,1.0,1.0,2.0,0.0,9.0,13.0
101,2.0,1.0,2.0,1.0,7.0,13.0
162,0.0,0.0,1.0,3.0,9.0,13.0
147,0.0,0.0,0.0,2.0,11.0,13.0
66,2.0,0.0,2.0,3.0,5.0,12.0
90,1.0,3.0,0.0,1.0,7.0,12.0
97,2.0,1.0,0.0,0.0,9.0,12.0
25,2.0,2.0,0.0,2.0,6.0,12.0


In [123]:
top_risk_stores = risk_pivot.head(15).reset_index()
top_risk_stores["store_id_dispatch"] = top_risk_stores["store_id_dispatch"].astype(str)

plot_df = top_risk_stores.sort_values("total_anomalias", ascending=True).copy()

fig = px.bar(
    plot_df,
    x="total_anomalias",
    y="store_id_dispatch",
    orientation="h",
    text="total_anomalias"
)

fig.update_traces(
    marker=dict(
        color=plot_df["total_anomalias"],
        colorscale=[
            [0.00, "#3B82F6"],
            [0.35, "#06B6D4"],
            [0.60, "#8B5CF6"],
            [1.00, "#EC4899"]
        ],
        line=dict(color="rgba(255,255,255,0.18)", width=1.2)
    ),
    textposition="outside",
    textfont=dict(color="white", size=15, family="Arial Black"),
    hovertemplate="<b>Tienda %{y}</b><br>Total anomalías: %{x}<extra></extra>"
)

fig.update_layout(
    title={
        "text": "Top 15 tiendas prioritarias por volumen de anomalías",
        "x": 0.03,
        "xanchor": "left",
        "font": dict(size=30, color="white", family="Arial Black")
    },
    template="plotly_dark",
    plot_bgcolor="#1E1E1E",
    paper_bgcolor="#1E1E1E",
    font=dict(color="white", size=14, family="Arial"),
    xaxis=dict(
        title="Total de anomalías",
        showgrid=True,
        gridcolor="rgba(255,255,255,0.08)",
        zeroline=False,
        title_font=dict(size=18, color="white"),
        tickfont=dict(size=13, color="white")
    ),
    yaxis=dict(
        title="Store ID",
        showgrid=False,
        categoryorder="total ascending",
        title_font=dict(size=18, color="white"),
        tickfont=dict(size=13, color="white")
    ),
    margin=dict(l=110, r=70, t=90, b=60),
    height=720,
    coloraxis_showscale=False
)

fig.add_annotation(
    text="Ranking de tiendas con mayor concentración de eventos anómalos en productos prioritarios",
    xref="paper", yref="paper",
    x=0.03, y=1.06,
    showarrow=False,
    font=dict(size=14, color="rgba(255,255,255,0.75)", family="Arial")
)

fig.show()

In [124]:
top_5_share = risk_pivot.head(5)["total_anomalias"].sum() / risk_pivot["total_anomalias"].sum() * 100
top_10_share = risk_pivot.head(10)["total_anomalias"].sum() / risk_pivot["total_anomalias"].sum() * 100

print(f"Participación del Top 5 tiendas: {top_5_share:.2f}%")
print(f"Participación del Top 10 tiendas: {top_10_share:.2f}%")

Participación del Top 5 tiendas: 5.00%
Participación del Top 10 tiendas: 9.47%


In [125]:
cols_investigativas = [
    "CATALOGO_INVISIBLE",
    "INCIDENCIA_OPERATIVA_DAÑO",
    "INCIDENCIA_OPERATIVA_FALTANTE",
    "INCIDENCIA_PROVEEDOR"
]

risk_investigative = risk_pivot[cols_investigativas].copy()
risk_investigative["total_investigativo"] = risk_investigative[cols_investigativas].sum(axis=1)
risk_investigative = risk_investigative.sort_values("total_investigativo", ascending=False)

risk_investigative.head(20)

anomaly_type,CATALOGO_INVISIBLE,INCIDENCIA_OPERATIVA_DAÑO,INCIDENCIA_OPERATIVA_FALTANTE,INCIDENCIA_PROVEEDOR,total_investigativo
store_id_dispatch,,,,,
66,2.0,0.0,2.0,3.0,7.0
135,0.0,4.0,1.0,1.0,6.0
101,2.0,1.0,2.0,1.0,6.0
5,2.0,1.0,2.0,1.0,6.0
25,2.0,2.0,0.0,2.0,6.0
94,3.0,2.0,0.0,1.0,6.0
9,1.0,0.0,2.0,3.0,6.0
159,2.0,1.0,1.0,2.0,6.0
52,0.0,1.0,1.0,4.0,6.0


In [126]:
top_5_inv = risk_investigative.head(5)["total_investigativo"].sum() / risk_investigative["total_investigativo"].sum() * 100
top_10_inv = risk_investigative.head(10)["total_investigativo"].sum() / risk_investigative["total_investigativo"].sum() * 100

print(f"Participación investigativa del Top 5 tiendas: {top_5_inv:.2f}%")
print(f"Participación investigativa del Top 10 tiendas: {top_10_inv:.2f}%")

Participación investigativa del Top 5 tiendas: 5.94%
Participación investigativa del Top 10 tiendas: 11.69%


In [127]:
merged_priority_units = merged_priority_named[
    merged_priority_named["unit_type"] == "UNIDAD"
].copy()

print("Registros prioritarios en UNIDAD:", len(merged_priority_units))
merged_priority_units[[
    "dispatch_id",
    "description_sap",
    "category_sap_dispatch",
    "quantity_dispatched",
    "quantity_received",
    "unit_type",
    "notes"
]].head(20)

Registros prioritarios en UNIDAD: 4505


,dispatch_id,description_sap,category_sap_dispatch,quantity_dispatched,quantity_received,unit_type,notes
0,DSP-00000158,"CAMARA HISENSE 75""",ELECTRONICA,8,8,UNIDAD,caja dañada
1,DSP-00000178,CONSOLA LG 128GB,ELECTRONICA,3,3,UNIDAD,OK
2,DSP-00000180,"MOUSE LENOVO 55""",COMPUTO,3,3,UNIDAD,producto sin codigo en sistema
3,DSP-00000242,MONITOR LG 256GB,ELECTRONICA,1,1,UNIDAD,revisar con proveedor
4,DSP-00000248,"WEBCAM LENOVO 32""",COMPUTO,3,3,UNIDAD,OK
5,DSP-00000293,"TELEVISOR SAMSUNG 55""",ELECTRONICA,9,9,UNIDAD,caja dañada
7,DSP-00000327,"BOCINA SONY 55""",ELECTRONICA,4,4,UNIDAD,NaN
8,DSP-00000352,TELEVISOR SAMSUNG 128GB,ELECTRONICA,5,5,UNIDAD,caja dañada
9,DSP-00000377,"PANTALLA JBL 65""",ELECTRONICA,6,6,UNIDAD,caja dañada
10,DSP-00000442,"PANTALLA JBL 65""",ELECTRONICA,4,4,UNIDAD,caja dañada


In [128]:
severity_by_store = (
    merged_priority_units
    .groupby("store_id_dispatch")[["quantity_dispatched", "quantity_received"]]
    .sum()
    .reset_index()
)

severity_by_store["quantity_diff"] = (
    severity_by_store["quantity_dispatched"] - severity_by_store["quantity_received"]
)

severity_by_store["pct_diff"] = np.where(
    severity_by_store["quantity_dispatched"] > 0,
    (severity_by_store["quantity_diff"] / severity_by_store["quantity_dispatched"]) * 100,
    np.nan
)

severity_by_store = severity_by_store.sort_values("pct_diff", ascending=False)

severity_by_store.head(20)

,store_id_dispatch,quantity_dispatched,quantity_received,quantity_diff,pct_diff
155,156,122,97,25,20.491803
46,47,148,118,30,20.270270
14,15,156,125,31,19.871795
82,83,83,69,14,16.867470
28,29,142,119,23,16.197183
133,134,136,115,21,15.441176
66,67,162,138,24,14.814815
90,91,170,145,25,14.705882
111,112,174,151,23,13.218391
170,171,206,179,27,13.106796


In [129]:
plot_df = severity_by_store.head(15).copy()
plot_df["store_id_dispatch"] = plot_df["store_id_dispatch"].astype(str)

fig = px.bar(
    plot_df.sort_values("pct_diff", ascending=True),
    x="pct_diff",
    y="store_id_dispatch",
    orientation="h",
    text="pct_diff"
)

fig.update_traces(
    marker=dict(
        color=plot_df.sort_values("pct_diff", ascending=True)["pct_diff"],
        colorscale=[
            [0.00, "#22C55E"],
            [0.35, "#EAB308"],
            [0.70, "#F97316"],
            [1.00, "#EF4444"]
        ],
        line=dict(color="rgba(255,255,255,0.18)", width=1.2)
    ),
    texttemplate="%{text:.1f}%",
    textposition="outside",
    textfont=dict(color="white", size=14, family="Arial Black"),
    hovertemplate="<b>Tienda %{y}</b><br>Diferencia porcentual: %{x:.2f}%<extra></extra>"
)

fig.update_layout(
    title={
        "text": "Top 15 tiendas por severidad de discrepancia (solo UNIDAD)",
        "x": 0.03,
        "xanchor": "left",
        "font": dict(size=28, color="white", family="Arial Black")
    },
    template="plotly_dark",
    plot_bgcolor="#1E1E1E",
    paper_bgcolor="#1E1E1E",
    font=dict(color="white", size=14, family="Arial"),
    xaxis=dict(
        title="% de diferencia entre despacho y recepción",
        showgrid=True,
        gridcolor="rgba(255,255,255,0.08)",
        zeroline=False
    ),
    yaxis=dict(
        title="Store ID",
        showgrid=False,
        categoryorder="total ascending"
    ),
    margin=dict(l=110, r=70, t=90, b=60),
    height=720,
    coloraxis_showscale=False
)

fig.show()

In [130]:
severity_by_store_category = (
    merged_priority_units
    .groupby(["store_id_dispatch", "category_sap_dispatch"])[["quantity_dispatched", "quantity_received"]]
    .sum()
    .reset_index()
)

severity_by_store_category["quantity_diff"] = (
    severity_by_store_category["quantity_dispatched"] - severity_by_store_category["quantity_received"]
)

severity_by_store_category["pct_diff"] = np.where(
    severity_by_store_category["quantity_dispatched"] > 0,
    (severity_by_store_category["quantity_diff"] / severity_by_store_category["quantity_dispatched"]) * 100,
    np.nan
)

severity_by_store_category.sort_values("pct_diff", ascending=False).head(30)

,store_id_dispatch,category_sap_dispatch,quantity_dispatched,quantity_received,quantity_diff,pct_diff
310,156,COMPUTO,48,37,11,22.916667
92,47,COMPUTO,22,17,5,22.727273
164,83,COMPUTO,40,31,9,22.500000
28,15,COMPUTO,62,49,13,20.967742
93,47,ELECTRONICA,126,101,25,19.841270
...,...,...,...,...,...,...
173,87,ELECTRONICA,78,76,2,2.564103
214,108,COMPUTO,39,38,1,2.564103
306,154,COMPUTO,40,39,1,2.500000
244,123,COMPUTO,42,41,1,2.380952


In [131]:
top_severe_stores = severity_by_store.head(10)["store_id_dispatch"].tolist()

severity_focus = severity_by_store_category[
    severity_by_store_category["store_id_dispatch"].isin(top_severe_stores)
].copy()

severity_focus["store_id_dispatch"] = severity_focus["store_id_dispatch"].astype(str)

fig = px.bar(
    severity_focus,
    x="store_id_dispatch",
    y="pct_diff",
    color="category_sap_dispatch",
    barmode="group",
    title="Severidad de discrepancia por tienda y categoría (Top 10 tiendas)",
    labels={
        "store_id_dispatch": "Store ID",
        "pct_diff": "% de diferencia",
        "category_sap_dispatch": "Categoría"
    }
)

fig.update_layout(
    template="plotly_dark",
    plot_bgcolor="#1E1E1E",
    paper_bgcolor="#1E1E1E",
    font=dict(color="white"),
    height=650
)

fig.show()

In [132]:
merged_priority_units["reception_hour"] = (
    pd.to_datetime(merged_priority_units["reception_time"], format="%H:%M", errors="coerce")
    .dt.hour
)

merged_priority_units[[
    "store_id_dispatch",
    "description_sap",
    "reception_time",
    "reception_hour"
]].head(20)

,store_id_dispatch,description_sap,reception_time,reception_hour
0,1,"CAMARA HISENSE 75""",09:56,9
1,1,CONSOLA LG 128GB,07:21,7
2,1,"MOUSE LENOVO 55""",09:04,9
3,1,MONITOR LG 256GB,07:48,7
4,1,"WEBCAM LENOVO 32""",08:03,8
5,1,"TELEVISOR SAMSUNG 55""",08:14,8
7,1,"BOCINA SONY 55""",07:11,7
8,1,TELEVISOR SAMSUNG 128GB,09:20,9
9,1,"PANTALLA JBL 65""",07:08,7
10,1,"PANTALLA JBL 65""",08:10,8


In [133]:
top_severe_store_ids = severity_by_store.head(10)["store_id_dispatch"].tolist()

hourly_top = merged_priority_units[
    merged_priority_units["store_id_dispatch"].isin(top_severe_store_ids)
]["reception_hour"].value_counts().sort_index()

hourly_rest = merged_priority_units[
    ~merged_priority_units["store_id_dispatch"].isin(top_severe_store_ids)
]["reception_hour"].value_counts().sort_index()

hourly_compare = pd.DataFrame({
    "Top tiendas severas": hourly_top,
    "Resto de tiendas": hourly_rest
}).fillna(0)

hourly_compare

,Top tiendas severas,Resto de tiendas
reception_hour,,
5,175,0.0
7,23,1418.0
8,21,1424.0
9,23,1421.0


In [134]:
hourly_compare.reset_index().columns

Index(['reception_hour', 'Top tiendas severas', 'Resto de tiendas'], dtype='str')

In [135]:
hourly_plot = hourly_compare.reset_index()

hourly_plot = hourly_plot.melt(
    id_vars="reception_hour",
    var_name="grupo",
    value_name="conteo"
)

fig = px.bar(
    hourly_plot,
    x="reception_hour",
    y="conteo",
    color="grupo",
    barmode="group",
    title="Distribución horaria de recepciones: tiendas severas vs resto",
    labels={
        "reception_hour": "Hora de recepción",
        "conteo": "Número de recepciones",
        "grupo": "Grupo"
    }
)

fig.update_layout(
    template="plotly_dark",
    plot_bgcolor="#1E1E1E",
    paper_bgcolor="#1E1E1E",
    font=dict(color="white"),
    height=650
)

fig.show()

In [136]:
top_severe_store_ids = severity_by_store.head(10)["store_id_dispatch"].tolist()

top_severe_store_meta = stores[
    stores["store_id"].isin(top_severe_store_ids)
][[
    "store_id",
    "store_name",
    "city",
    "state",
    "system",
    "cedis_route",
    "regional_manager_id",
    "ghost_store"
]].sort_values("store_id")

top_severe_store_meta

,store_id,store_name,city,state,system,cedis_route,regional_manager_id,ghost_store
14,15,NOVA-SAN-015,San Luis Potosí,San Luis Potosí,SAP,NORTE,GR-001,False
28,29,NOVA-SAN-029,San Nicolás,Nuevo León,SAP,NORTE,GR-005,False
46,47,NOVA-MAZ-047,Mazatlán,Sinaloa,SAP,NORTE,GR-002,False
66,67,NOVA-TAM-067,Tampico,Tamaulipas,SAP,NORTE,GR-003,False
82,83,NOVA-MEX-083,Mexicali,Baja California,SAP,NORTE,GR-002,False
90,91,NOVA-SAL-091,Salamanca,Guanajuato,SAP,NORTE,GR-002,False
111,112,NOVA-OBR-112,Obregón,Sonora,SAP,NORTE,GR-002,False
133,134,NOVA-CUL-134,Culiacán,Sinaloa,AS400,NORTE,GR-005,False
155,156,NOVA-OBR-156,Obregón,Sonora,AS400,NORTE,GR-002,False
170,171,NOVA-MAZ-171,Mazatlán,Sinaloa,AS400,NORTE,GR-002,True


In [137]:
top_severe_store_meta["cedis_route"].value_counts()

cedis_route
NORTE    10
Name: count, dtype: int64

In [138]:
top_severe_store_meta["regional_manager_id"].value_counts()

regional_manager_id
GR-002    6
GR-005    2
GR-001    1
GR-003    1
Name: count, dtype: int64

In [139]:
early_hour_counts = (
    merged_priority_units[merged_priority_units["reception_hour"] == 5]
    .groupby("store_id_dispatch")
    .size()
    .reset_index(name="recepciones_5am")
    .sort_values("recepciones_5am", ascending=False)
)

early_hour_counts.head(20)

,store_id_dispatch,recepciones_5am
0,15,21
9,171,21
2,47,20
3,67,20
8,156,18
1,29,17
5,91,17
7,134,16
6,112,15
4,83,10


In [140]:
top_severe_5am = severity_by_store.merge(
    early_hour_counts,
    left_on="store_id_dispatch",
    right_on="store_id_dispatch",
    how="left"
).fillna({"recepciones_5am": 0})

top_severe_5am = top_severe_5am.sort_values("pct_diff", ascending=False)

top_severe_5am.head(15)

,store_id_dispatch,quantity_dispatched,quantity_received,quantity_diff,pct_diff,recepciones_5am
0,156,122,97,25,20.491803,18.0
1,47,148,118,30,20.270270,20.0
2,15,156,125,31,19.871795,21.0
3,83,83,69,14,16.867470,10.0
4,29,142,119,23,16.197183,17.0
5,134,136,115,21,15.441176,16.0
6,67,162,138,24,14.814815,20.0
7,91,170,145,25,14.705882,17.0
8,112,174,151,23,13.218391,15.0
9,171,206,179,27,13.106796,21.0


In [141]:
plot_final = top_severe_5am.head(15).copy()
plot_final["store_id_dispatch"] = plot_final["store_id_dispatch"].astype(str)

fig = px.scatter(
    plot_final,
    x="recepciones_5am",
    y="pct_diff",
    size="quantity_diff",
    color="pct_diff",
    text="store_id_dispatch",
    color_continuous_scale=[
        [0.00, "#22C55E"],
        [0.35, "#EAB308"],
        [0.70, "#F97316"],
        [1.00, "#EF4444"]
    ],
    title="Relación entre severidad de discrepancia y recepciones a las 5 AM"
)

fig.update_traces(
    textposition="top center",
    marker=dict(line=dict(color="rgba(255,255,255,0.25)", width=1.2)),
    hovertemplate=(
        "<b>Tienda %{text}</b><br>" +
        "Recepciones 5 AM: %{x}<br>" +
        "Diferencia %: %{y:.2f}<br>" +
        "Diferencia absoluta: %{marker.size}<extra></extra>"
    )
)

fig.update_layout(
    template="plotly_dark",
    plot_bgcolor="#1E1E1E",
    paper_bgcolor="#1E1E1E",
    font=dict(color="white", size=14),
    height=700,
    xaxis_title="Número de recepciones a las 5 AM",
    yaxis_title="% de diferencia entre despacho y recepción",
    coloraxis_showscale=False
)

fig.show()

## Cierre ejecutivo — Versión CEO

**Conclusión ejecutiva:**  
El análisis confirma que las discrepancias más severas no están distribuidas al azar: se concentran en un grupo específico de tiendas dentro de la Ruta Norte y coinciden con una ventana de recepción atípica a las 5 AM.

**Implicación estratégica:**  
Ese patrón no es consistente con fricción operativa normal y sugiere una vulnerabilidad localizada en la cadena de custodia.

**Recomendación inmediata:**  
Avanzar de inmediato a una investigación focalizada sobre ese corredor logístico antes de que el patrón se adapte, se opaque o desaparezca.

## Cierre ejecutivo — Versión CFO

**Conclusión ejecutiva:**  
La pérdida no parece estar dispersa en toda la red, sino concentrada en un subconjunto operativo con discrepancias de recepción de entre 13% y 20.5% sobre productos prioritarios.

**Implicación financiera:**  
Esto permite abandonar una limpieza horizontal costosa y migrar hacia una intervención focalizada sobre las tiendas y flujos que realmente están deteriorando valor.

**Recomendación inmediata:**  
Concentrar recursos analíticos y operativos en la Ruta Norte, donde la severidad observada supera ampliamente el rango de merma operativa esperada y ofrece la mejor relación entre esfuerzo e impacto económico.

## Cierre ejecutivo — Versión Legal

**Conclusión ejecutiva:**  
Los datos identifican una anomalía operativa localizada caracterizada por discrepancias materiales entre despacho y recepción, asociadas a una franja horaria atípica y a un corredor logístico específico.

**Límite probatorio actual:**  
Estos hallazgos no constituyen por sí mismos prueba de conducta individual indebida, pero sí establecen una base objetiva para una revisión focalizada de controles, registros y cadena de custodia.

**Recomendación inmediata:**  
Cualquier escalamiento a personas, responsabilidades o medidas disciplinarias deberá apoyarse en una segunda fase de evidencia nominal y en un protocolo formal de preservación de evidencia.

In [142]:
resumen_dispatch_reception = pd.DataFrame({
    "indicador": [
        "Despachos totales",
        "Recepciones totales",
        "Registros prioritarios analizados",
        "Tiendas severas identificadas",
        "Máxima discrepancia porcentual",
        "Mínima discrepancia en tiendas normales",
        "Tiendas severas con recepciones 5 AM",
        "Ruta dominante en tiendas severas"
    ],
    "valor": [
        len(dispatches),
        len(receptions),
        len(merged_priority_units),
        10,
        f"{severity_by_store['pct_diff'].max():.2f}%",
        f"{severity_by_store['pct_diff'].tail(5).min():.2f}%",
        int((top_severe_5am.head(10)["recepciones_5am"] > 0).sum()),
        "NORTE"
    ]
})

resumen_dispatch_reception

,indicador,valor
0,Despachos totales,265964
1,Recepciones totales,265964
2,Registros prioritarios analizados,4505
3,Tiendas severas identificadas,10
4,Máxima discrepancia porcentual,20.49%
5,Mínima discrepancia en tiendas normales,0.00%
6,Tiendas severas con recepciones 5 AM,10
7,Ruta dominante en tiendas severas,NORTE


## Conclusiones de la fase despacho vs recepción

### Hallazgos principales
- El cruce entre despachos y recepciones mostró integridad estructural a nivel de llave (`dispatch_id` ↔ `dispatch_reference`).
- La métrica más útil para detectar el patrón relevante no fue la frecuencia de incidencias, sino la **severidad porcentual de discrepancia**.
- En productos prioritarios registrados como `UNIDAD`, emergió un subconjunto de tiendas con diferencias de recepción entre **13% y 20.5%**, muy por encima del rango operativo normal.
- Las tiendas con mayor severidad se concentran completamente en la **Ruta Norte**.
- Ese mismo subconjunto presenta recepciones en la franja de las **05:00**, ausente en las tiendas de comportamiento normal.
- La combinación de **severidad + ruta + horario** constituye una señal operativa localizada e incompatible con ruido distribuido.

### Límite actual
Este notebook no atribuye responsabilidad individual ni prueba fraude por sí mismo. Su función es identificar un patrón operativo priorizable y defendible para escalamiento investigativo.

### Decisión metodológica
La siguiente fase debe concentrarse en profundizar el análisis sobre la Ruta Norte, integrando:
- trazabilidad de usuarios
- metadata de autorización
- comparación por tienda y corredor logístico
- y evidencia nominal bajo control de cadena de custodia

## Próximo paso recomendado

Con base en los hallazgos de esta fase, la siguiente etapa del análisis debe enfocarse en:

1. **Cruzar severidad de discrepancia con usuarios autorizadores y metadatos de despacho**
2. **Analizar la relación entre ruta, horario de recepción y pérdida material**
3. **Separar de forma controlada ruido operativo, error de unidad de medida y señales de vulnerabilidad localizada**
4. **Preparar una matriz de priorización investigativa para escalamiento ejecutivo**

El análisis deja claro que la siguiente pregunta ya no es si existe una anomalía, sino **quiénes y qué controles están posicionados alrededor de ella**.